# Run simulations to show effect of number of arrivals

## Load libraries

In [ ]:
import xml.etree.ElementTree as ET
import libs.tools as tools
from libs.socket_connect import QS_socket_connect as QS

## Configure socket connect

(Since we are running many, small simulation, we use a socket connection here. -- Instead of starting and stopping the JVM for each run.)

In [ ]:
setup = {
    "host": "localhost",
    "port": 10000
}

## Load base model

In [ ]:
base_file = "Models_Base/RunCount.xml"

_, model_root = tools.load_xml_file(base_file)

## Helper function for changing the number of arrivals

In [ ]:
def set_number_of_arrivals(root, count: int, file: str):
    el = tools.get_element(root, 'ModelClients', file)
    el.text = str(count)

## Run simulations and store results

In [ ]:
socket = QS(setup["host"], setup["port"])

for arrival_count_exp in [6, 7, 8, 9, 10]:
    print(f"10^{arrival_count_exp} arrivals")
    arrival_count = 10**arrival_count_exp
    set_number_of_arrivals(model_root, arrival_count, base_file)
    model_str = ET.tostring(model_root, encoding='utf-8', method='xml')

    for i in range(100):
        if i % 10 == 0:
            print(f"  {i}%")
        statistics_str = socket.run_task(model_str)
        statistics_tree = ET.ElementTree(ET.fromstring(statistics_str))
        statistics_root = statistics_tree.getroot()
        mean = statistics_root.find("QS:StatisticsWaitingTimesAllClients", tools.ns).attrib["Mean"] # type: ignore

        with open(f'Statistics/WaitingTimes-{arrival_count_exp}.txt', 'a') as file:
            file.write(mean+"\n")